
# ElasticNet Pipeline Test Notebook

This notebook lets you run your `src/pipeline.py` on a **controllable list of proteins** and execute all three pipeline styles:

- `fixed_percentile` (via `pipeline.py` CLI)
- `fixed_p` (via `pipeline.py` CLI)
- `fixed_features` (**workaround**: run directly through `pipeline_functions.cv_pipeline_fixed_features` because the current `pipeline.py` argparse choices omit `'fixed_features'`.)

> **Note:** In `pipeline.py`, the defaults are hard-coded to `percentile=5`, `p_value=0.05`, and `n_splits=5` when called via CLI.  
> If you want to change those values, either edit `src/pipeline.py` or run the direct helpers provided here (used for `fixed_features`).

Folder expectations (as in your screenshot):
```
pipeline_proteins/
├─ data/
│  ├─ transcriptomics_filtered.joblib
│  └─ proteomics_matched.joblib
├─ src/
│  ├─ pipeline.py
│  └─ pipeline_functions.py
└─ enet_pipeline_test.ipynb   ← this notebook
```


In [1]:

# --- Setup: resolve project root & PYTHONPATH so imports work ---
import os, sys, subprocess, json, textwrap
from pathlib import Path

#other imports
import sys
import subprocess
from joblib import load, dump
import pandas as pd
from sklearn.linear_model import ElasticNet


# Try to locate project root that contains /src and /data
CWD = Path.cwd()
CANDIDATES = [CWD, CWD.parent, CWD.parent.parent]

PROJ_ROOT = None
for c in CANDIDATES:
    if (c / "src" / "pipeline.py").exists() and (c / "data").exists():
        PROJ_ROOT = c.resolve()
        break

if PROJ_ROOT is None:
    raise FileNotFoundError(
        "Couldn't find project root. Make sure you're running this notebook "
        "from the repository that contains 'src/pipeline.py' and a 'data' folder."
    )

print(f'Project root resolved to: {PROJ_ROOT}')

SRC_DIR = PROJ_ROOT / "src"
DATA_DIR = PROJ_ROOT / "data"

# Ensure we can import from src/
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# Light sanity checks (won't error if missing, just informative)
print("Found files:")
print(" - src/pipeline.py:", (SRC_DIR / "pipeline.py").exists())
print(" - src/pipeline_functions.py:", (SRC_DIR / "pipeline_functions.py").exists())
print(" - data/transcriptomics_filtered.joblib:", (DATA_DIR / "transcriptomics_filtered.joblib").exists())
print(" - data/proteomics_matched.joblib:", (DATA_DIR / "proteomics_matched.joblib").exists())

# Direct import for the fixed_features workaround
from pipeline_functions import cv_pipeline_fixed_features


Project root resolved to: C:\Users\hp\Desktop\CPPR\Cancer-Protein-RNA-research\03. elasticNet\pipeline_proteins
Found files:
 - src/pipeline.py: True
 - src/pipeline_functions.py: True
 - data/transcriptomics_filtered.joblib: True
 - data/proteomics_matched.joblib: True


In [2]:

# --- User configuration ---
# Provide the list of proteins you'd like to run.
# Example: proteins = ["ADH1A", "GAPDH", "TP53"]
proteins = [
    # <<< PUT YOUR PROTEIN NAMES HERE >>>
    "A2M"
]

# Pipelines to run; you can subset this list as needed.
#pipelines_to_run = ["fixed_percentile", "fixed_p", "fixed_features"]
pipelines_to_run = ["fixed_percentile"]
#pipelines_to_run = ["fixed_p"]
#pipelines_to_run = ["fixed_features"]


# Model choice (your pipeline.py supports 'elasticnet' and 'xgboost', but code currently sets ElasticNet)
model = "elasticnet"

# For the direct `fixed_features` run below, we allow optional overrides.
# (These are *not* used by the CLI, which keeps its own defaults.)
n_splits_direct = 5
param_grid_direct = {
    "alpha": [0.1, 0.5, 1, 10],
    "l1_ratio": [0.1, 0.5, 0.9],
}

# Optional: limit number of samples for a quick test (None means use all)
n_samples_direct = 50  # e.g., 50


In [3]:
# Helper functions

def run_pipeline_cli(protein: str, pipeline_name: str, model: str = "elasticnet") -> int:
    """Run src/pipeline.py as a script via subprocess for the supported pipelines.
    Returns the subprocess return code.
    """
    script_path = SRC_DIR / "pipeline.py"
    cmd = [sys.executable, str(script_path),
           "--protein", protein,
           "--pipeline", pipeline_name,
           "--model", model]
    print(">>> Running:", " ".join(cmd))
    proc = subprocess.run(cmd, cwd=str(SRC_DIR.parent))  # run from project root to be safe
    if proc.returncode != 0:
        print(f"[WARN] pipeline.py returned non-zero exit code ({proc.returncode}) for {protein} | {pipeline_name}")
    else:
        print(f"[OK] Completed {protein} | {pipeline_name}")
    return proc.returncode


def run_fixed_features_direct(protein: str,
                              param_grid: dict,
                              n_splits: int = 5,
                              n_samples=None):
    """Workaround runner for 'fixed_features' since pipeline.py argparse omits it.
    Mirrors pipeline.py behavior & output path.
    """
    # Load data
    transcriptomics_file = DATA_DIR / "transcriptomics_filtered.joblib"
    proteomics_file = DATA_DIR / "proteomics_matched.joblib"
    if not transcriptomics_file.exists() or not proteomics_file.exists():
        raise FileNotFoundError("Missing required data files in 'data/' folder.")

    X = load(transcriptomics_file)
    Y = load(proteomics_file)
    X.index.name = "samples"
    Y.index.name = "samples"
    Y.columns.name = "proteins"
    X.columns.name = "genes"

    if protein not in Y.columns:
        raise ValueError(f"Protein '{protein}' not found in proteomics data.")

    # Load per-sample feature selections
    fs_path = DATA_DIR / f"{protein}_feature_selection_LOO.joblib"
    if not fs_path.exists():
        raise FileNotFoundError(
            f"Expected feature selection file not found: {fs_path}\n"
            "Generate it first or adjust the path."
        )
    features_df = load(fs_path)

    # Build and run the CV
    model = ElasticNet()
    results_df = cv_pipeline_fixed_features(model=model,
                                            param_grid=param_grid,
                                            X=X,
                                            y=Y,
                                            protein_name=protein,
                                            features=features_df,
                                            n_splits=n_splits,
                                            n_samples=n_samples)

    # Save with the same naming pattern as pipeline.py
    out_path = DATA_DIR / f"{protein}_fixed_features_elasticnet_LOO_results.joblib"
    dump(results_df, out_path)
    print(f"[OK] Saved results to {out_path}")
    return out_path


In [ ]:

# --- Execute ---
if not proteins:
    print("No proteins specified. Edit the 'proteins' list above and re-run this cell.")
else:
    for prot in proteins:
        for pipe in pipelines_to_run:
            if pipe in {"fixed_percentile", "fixed_p"}:
                run_pipeline_cli(protein=prot, pipeline_name=pipe, model=model)
            elif pipe == "fixed_features":
                try:
                    run_fixed_features_direct(
                        protein=prot,
                        param_grid=param_grid_direct,
                        n_splits=n_splits_direct,
                        n_samples=n_samples_direct
                    )
                except Exception as e:
                    print(f"[ERROR] fixed_features failed for {prot}: {e}")
            else:
                print(f"[WARN] Unknown pipeline: {pipe}")


>>> Running: c:\Users\hp\anaconda3\python.exe C:\Users\hp\Desktop\CPPR\Cancer-Protein-RNA-research\03. elasticNet\pipeline_proteins\src\pipeline.py --protein A2M --pipeline fixed_percentile --model elasticnet


In [ ]:

# check produced result files

result_paths = sorted(DATA_DIR.glob("*_LOO_results.joblib"))
if not result_paths:
    print("No results found yet in data/. After running, you'll see *_LOO_results.joblib files here.")
else:
    print("Found result files:")
    for p in result_paths:
        print(" -", p.name)
    # Try loading the first one for a quick peek
    try:
        df = load(result_paths[0])
        print("\nPreview of the first results file:")
        print(df.head())
    except Exception as e:
        print(f"Couldn't preview {result_paths[0].name}: {e}")
